# NetworkX: Research Collaboration Networks

This notebook introduces NetworkX through a **small made-up dataset** that mirrors real EU funding data. Work through each section to understand how to build, style, and analyse a collaboration network — then attempt the **Your Turn** exercise at the end using real CORDIS data.

## Contents
1. Small example dataset  
2. Basic network graph  
3. Styled visualisation (edge width, node size, colour)  
4. Community detection  
5. Centrality analysis  
6. **Your Turn** — build the same network with real CORDIS data

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from itertools import combinations
from collections import Counter

from template import setup_matplotlib_template, get_colors

# Apply TG house style (font, colours, layout)
setup_matplotlib_template('TG')
TG_COLORS = get_colors('TG')

---
## 1. Small Example Dataset

We create a tiny dataset that mimics the structure of CORDIS EU funding records. Each row represents one organisation's participation in one project.

| Column | Description |
|---|---|
| `projectID` | The project |
| `orgName` | Organisation name — used as the graph node |
| `country` | 2-letter country code |
| `orgType` | Type of organisation — used for node colour |
| `ecContribution` | EC funding received for this project (€) |

In [39]:
# Small made-up EU research collaboration dataset
data = {
    'projectID':      [1, 1, 1,  2, 2,         3, 3, 3,         4, 4,         5, 5, 5, 5],
    'orgName':        ['TU Berlin', 'CNRS', 'Siemens',
                       'TU Berlin', 'Fraunhofer',
                       'CNRS', 'UCL', 'Philips',
                       'Fraunhofer', 'Siemens',
                       'UCL', 'TU Berlin', 'CNRS', 'Fraunhofer'],
    'country':        ['DE', 'FR', 'DE',  'DE', 'DE',  'FR', 'GB', 'NL',  'DE', 'DE',  'GB', 'DE', 'FR', 'DE'],
    'orgType':        ['University', 'Research', 'Industry',
                       'University', 'Research',
                       'Research', 'University', 'Industry',
                       'Research', 'Industry',
                       'University', 'University', 'Research', 'Research'],
    'ecContribution': [500_000, 300_000, 0,
                       450_000, 600_000,
                       350_000, 400_000, 0,
                       550_000, 0,
                       420_000, 480_000, 320_000, 580_000],
}

df = pd.DataFrame(data)
df

,projectID,orgName,country,orgType,ecContribution
0,1,TU Berlin,DE,University,500000
1,1,CNRS,FR,Research,300000
2,1,Siemens,DE,Industry,0
3,2,TU Berlin,DE,University,450000
4,2,Fraunhofer,DE,Research,600000
5,3,CNRS,FR,Research,350000
6,3,UCL,GB,University,400000
7,3,Philips,NL,Industry,0
8,4,Fraunhofer,DE,Research,550000
9,4,Siemens,DE,Industry,0


---
## 2. Basic Network Graph

**Nodes** = organisations. **Edges** = two orgs participated in the same project.

Steps:
1. Compute per-org totals (EC funding, org type)
2. Build the collaboration edge list — one row per pair per project
3. Count how many projects each pair shared → `collaborations` edge weight
4. Draw the basic graph

In [20]:
# Per-org node attributes
org_attrs = (df.groupby('orgName')
               .agg(country=('country', 'first'),
                    orgType=('orgType', 'first'),
                    total_ec=('ecContribution', 'sum'))
               .reset_index())
print(org_attrs.to_string(index=False))

   orgName country    orgType  total_ec
      CNRS      FR   Research    970000
Fraunhofer      DE   Research   1730000
   Philips      NL   Industry         0
   Siemens      DE   Industry         0
 TU Berlin      DE University   1430000
       UCL      GB University    820000


In [32]:
# Collaboration edge list
edge_records = []
for pid, group in df.groupby('projectID'):
    orgs = group['orgName'].tolist()
    for a, b in combinations(sorted(orgs), 2):
        edge_records.append((a, b))

print(edge_records)

[('CNRS', 'Siemens'), ('CNRS', 'TU Berlin'), ('Siemens', 'TU Berlin'), ('Fraunhofer', 'TU Berlin'), ('CNRS', 'Philips'), ('CNRS', 'UCL'), ('Philips', 'UCL'), ('Fraunhofer', 'Siemens'), ('CNRS', 'Fraunhofer'), ('CNRS', 'TU Berlin'), ('CNRS', 'UCL'), ('Fraunhofer', 'TU Berlin'), ('Fraunhofer', 'UCL'), ('TU Berlin', 'UCL')]


In [37]:
edge_df = (pd.DataFrame(edge_records, columns=['org_a', 'org_b'])
             .groupby(['org_a', 'org_b'])
             .size()
             .reset_index(name='collaborations'))
print("Edge list: ", len(edge_df), "pairs")
print(edge_df.to_string(index=False))

Edge list:  11 pairs
     org_a      org_b  collaborations
      CNRS Fraunhofer               1
      CNRS    Philips               1
      CNRS    Siemens               1
      CNRS  TU Berlin               2
      CNRS        UCL               2
Fraunhofer    Siemens               1
Fraunhofer  TU Berlin               2
Fraunhofer        UCL               1
   Philips        UCL               1
   Siemens  TU Berlin               1
 TU Berlin        UCL               1


In [22]:
# Build NetworkX graph
G = nx.from_pandas_edgelist(edge_df, 'org_a', 'org_b', edge_attr='collaborations')
nx.set_node_attributes(G, org_attrs.set_index('orgName').to_dict('index'))
print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Graph: 6 nodes, 11 edges


In [38]:
G

In [ ]:
# Basic draw
pos = nx.spring_layout(G, seed=42, k=1)
fig, ax = plt.subplots(figsize=(11, 8))
nx.draw(G, pos, ax=ax, with_labels=True,
        node_color=TG_COLORS[1],   # TG blue
        font_color='k', font_weight='bold',
        edge_color='#C4C4C4', width=1.5)
ax.set_title("Basic Collaboration Network")
plt.tight_layout()
plt.show()

---
## 3. Styled Visualisation

We now encode data directly into the visual properties:

| Visual property | Data |
|---|---|
| **Edge width** | `collaborations` — number of shared projects |
| **Node size** | `total_ec` — total EC funding received |
| **Node colour** | `orgType` — University / Research / Industry |

In [ ]:
# Edge widths — scaled by collaboration count
collab_counts = nx.get_edge_attributes(G, 'collaborations')
max_collab = max(collab_counts.values())
edge_widths = [1 + 5 * (collab_counts[e] / max_collab) for e in G.edges()]

# Node sizes — scaled by total EC funding
funding = {n: G.nodes[n].get('total_ec', 0) for n in G.nodes()}
max_funding = max(funding.values()) if max(funding.values()) > 0 else 1
node_sizes = [300 + 1700 * (funding[n] / max_funding) for n in G.nodes()]

# Node colours — by org type (using TG palette)
type_colors = {
    'University': TG_COLORS[1],   # TG blue
    'Research':   TG_COLORS[5],   # TG green
    'Industry':   TG_COLORS[2],   # TG yellow
}
node_colors = [type_colors.get(G.nodes[n].get('orgType'), '#C4C4C4') for n in G.nodes()]


fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
                       linewidths=0.5, edgecolors='white', ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='#C4C4C4', alpha=0.6, ax=ax)
nx.draw_networkx_labels(G, pos, font_color='k', font_weight='bold', font_size=9, ax=ax)

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_patches, title='Org Type', loc='upper left')
ax.set_title("Styled Collaboration Network\n"
             "(Node size = EC funding  |  Edge width = collaborations  |  Colour = org type)")
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Your Turn — CORDIS Collaboration Network

Now build the same network using **real EU funding data** from `cordis_orgs.csv`.

### Instructions

**1. Load the data**
```python
df_cordis = pd.read_csv('cordis_orgs.csv')
```

**2. Filter to a single master call**
```python
df_cordis = df_cordis[df_cordis['masterCall'] == 'HORIZON-CL5-2022-D2-01']
```
> This is the *Clean Energy* cluster from the 2022 Horizon Europe work programme — a good size for visualisation.

**3. Extract country from `vatNumber`**  
The column starts with a 2-letter country code (e.g. `FR45180089047` → `FR`). Use `.str.extract(r'^([A-Z]{2})')`.

**4. Compute per-org totals**  
Group by `organisationID` to get total `ecContribution` and first-occurrence `country`.  
Use `shortName` as the label (fall back to truncated `name` if missing).

**5. Build the edge list**  
For each `projectID`, create a link between every pair of organisations using `itertools.combinations`. Count how many projects each pair shared → store as `collaborations`.

**6. Build the graph and visualise**
- Start with a basic graph
- Add edge width, node size, and node colour
- Run community detection
- Run centrality analysis

> **Hint:** The full graph may have hundreds of nodes. Subgraph to the top 50–60 orgs by degree for a readable layout.  
> **Hint:** Some orgs have `ecContribution = 0` or `NaN` — handle these with a fallback (e.g. `or 0`).